In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

fake['label'] = 0
true['label'] = 1

df = pd.concat([fake, true], ignore_index=True)

print(df.shape)
df.head()


In [ ]:
df.info()
df.isnull().sum()


In [ ]:
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates()


In [ ]:
sns.countplot(x='label', data=df)
plt.show()


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['content'] = df['title'].astype(str) + ' ' + df['text'].astype(str)
df['content'] = df['content'].apply(clean_text)


In [ ]:
X = df['content']
y = df['label']

tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
X_tfidf = tfidf.fit_transform(X)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
model = LinearSVC()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test,pred))
print(classification_report(y_test,pred))


In [ ]:
cm = confusion_matrix(y_test,pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.show()


In [ ]:
param_grid = {'C':[0.01,0.1,1,10,100]}

grid = GridSearchCV(
    LinearSVC(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train,y_train)

print(grid.best_params_)


In [ ]:
final_model = grid.best_estimator_

final_pred = final_model.predict(X_test)

print('Final Accuracy:', accuracy_score(y_test,final_pred))
print(classification_report(y_test,final_pred))


In [ ]:
joblib.dump(final_model,'fake_news_detector.pkl')
joblib.dump(tfidf,'tfidf_vectorizer.pkl')
print('Saved')
